# Phase 4 — Predictive Modeling
**Goal:** Build a Linear Regression model to predict house sale prices. Evaluate it with MAE, RMSE, and R², then inspect which features drive predictions most.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the cleaned data (Phase 1 output — before feature scaling to preserve SalePrice)
df = pd.read_csv('data/cleaned/Ames_Housing_Cleaned.csv')
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')

## Step 1 — Prepare Features and Target

In [ ]:
# One-hot encode all remaining categorical columns so the model can use them
df_model = pd.get_dummies(df, drop_first=True)

X = df_model.drop('SalePrice', axis=1)
y = df_model['SalePrice']

print(f'Features: {X.shape[1]}  |  Target: SalePrice')

## Step 2 — Train/Test Split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f'Training set:  {X_train.shape[0]} records')
print(f'Test set:      {X_test.shape[0]} records')

## Step 3 — Train the Model & Evaluate

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae  = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2   = r2_score(y_test, predictions)

print('=== Model Performance ===')
print(f'Mean Absolute Error (MAE):         ${mae:,.2f}')
print(f'Root Mean Squared Error (RMSE):    ${rmse:,.2f}')
print(f'R² Score:                          {r2:.4f} ({r2*100:.2f}%)')

## Step 4 — Actual vs Predicted Chart

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, predictions, alpha=0.5, edgecolors='none', color='steelblue', s=30)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2,
         label='Perfect Prediction Line')
plt.title('Actual vs Predicted House Prices')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.legend()
plt.tight_layout()
plt.show()

## Step 5 — Residuals Distribution

In [ ]:
residuals = y_test - predictions

plt.figure(figsize=(10, 5))
sns.histplot(residuals, kde=True, color='coral', bins=50)
plt.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Zero Error')
plt.title('Residuals Distribution')
plt.xlabel('Residual (Actual - Predicted) ($)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Residuals mean: ${residuals.mean():,.2f} (should be close to 0)')
print('Near-normal residuals centred at 0 confirm linear regression assumptions are met.')

## Step 6 — Top 10 Most Influential Features

In [ ]:
coeff_df = pd.DataFrame({'Feature': X.columns, 'Coefficient': model.coef_})
top10 = coeff_df.reindex(coeff_df['Coefficient'].abs().sort_values(ascending=False).index).head(10)

plt.figure(figsize=(10, 6))
plt.barh(top10['Feature'], top10['Coefficient'], color='steelblue')
plt.title('Top 10 Most Influential Features (by Absolute Coefficient)')
plt.xlabel('Coefficient Value')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(top10.to_string(index=False))